# Clayton Metang — Expedition
High-level workflow using `expedition.py`. Each cell calls one stage; `x.save()` persists config after each step.

In [5]:
%load_ext autoreload
%autoreload 2
import logging
import claytonlib as clayton
from claytonlib.expedition import expedition

# --- Logging ---
# INFO shows per-write-cycle timing; DEBUG adds per-turn RNG details
logging.basicConfig(level=logging.INFO)
# logging.getLogger('claytonlib').setLevel(logging.INFO)

x = expedition("metang")
x.reload()
x.print()
x.chart_options.evaluation_frames_per_write_cycle = 5

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
[expedition] Reloaded from data/expeditions/metang.json
=== Expedition: metang ===
  pokemon                  metang
  key_seed                 0x0C0E02C2
  setup_delay_s            180
  max_target_s             600
  strategy                 six-bait-then-balls
  criteria                 machete-50-turns-after-5-balls
  eval_strategy            sliding_window_13
  window                   120
  target_delay             33805
  initial_time             2000-07-24T14:45:55
  target_seeds             ['0x1C1562D2']
  metronome_histsz         10
  metronome_second_window  2
  compass_m_delay          2441
  ---
  delay_from_key           33099 frames  (551.65s)


In [16]:
x.adjust(strategy_name="six-bait-then-balls")

[expedition] strategy_name = 'six-bait-then-balls'


# Chart - Finding a target

## Create chart

In [6]:
x.precompute_chart()
x.save()

[expedition] 10:17:24  === precompute_chart ===  (2026-09-09)
[expedition] 10:17:24  charting metang key_seed=0x0C0E02C2 delay=180-600s strategy=six-bait-then-balls criteria=machete-50-turns-after-5-balls  workers=12
[expedition] 10:17:40  mdmsh 1/248  elapsed 0.3m  eta ~66.1m
[expedition] 10:18:36  mdmsh 5/248  elapsed 1.2m  eta ~58.5m
[expedition] 10:19:45  mdmsh 10/248  elapsed 2.4m  eta ~56.2m
[expedition] 10:20:56  mdmsh 15/248  elapsed 3.5m  eta ~54.8m
[expedition] 10:22:06  mdmsh 20/248  elapsed 4.7m  eta ~53.7m
[expedition] 10:23:18  mdmsh 25/248  elapsed 5.9m  eta ~52.6m
[expedition] 10:24:31  mdmsh 30/248  elapsed 7.1m  eta ~51.8m
[expedition] 10:25:43  mdmsh 35/248  elapsed 8.3m  eta ~50.6m
[expedition] 10:26:34  mdmsh 40/248  elapsed 9.2m  eta ~47.7m
[expedition] 10:27:37  mdmsh 45/248  elapsed 10.2m  eta ~46.1m
[expedition] 10:28:46  mdmsh 50/248  elapsed 11.4m  eta ~45.0m
[expedition] 10:29:53  mdmsh 55/248  elapsed 12.5m  eta ~43.9m
[expedition] 10:31:02  mdmsh 60/248  e

## Evaluate chart to find targets

`chart_report()` ranks the best **(boot time, commanded countdown M)** pairs across all candidate boot times (mode A), or the best M for a boot time you pass as `initial_time=` (mode B). It saves its ranked findings so `select_target()` can use them.

In [7]:
x.chart_report()
x.save()

[expedition] 11:17:41  === chart_report ===
Best (boot time, M) pairs  [top 10 of 1668]  (jitter kernel, k=3.5):
   #            boot time     M (ms)  target F_b  second  P(capture)   sigma
   1  2000-05-31 14:54:59     176200       10941     181       29.3%    52.7
   2  2000-05-30 14:59:59     244860       15061     250       29.3%    62.2
   3  2000-01-01 14:00:11     195265       12085     201       29.0%    55.5
   4  2000-06-26 14:53:59     327919       20045     333       28.7%    71.9
   5  2000-05-30 14:59:59     412178       25101     417       28.4%    80.7
   6  2000-05-30 14:59:59     402179       24501     407       28.3%    79.7
   7  2000-01-01 14:00:11     268191       16461     273       28.3%    65.1
   8  2000-05-31 14:54:59     180066       11173     185       28.3%    53.3
   9  2000-07-29 14:55:10     463507       28181     469       28.2%    85.5
  10  2000-07-28 14:58:14     279457       17137     285       28.2%    66.4
[expedition] 11:17:47  best target for e

## Choose Target

`select_target()` reads the findings `chart_report()` saved and lets you pick one. It records the chosen **boot time** (`initial_time`), **timer countdown** (`target_timer_delay` = M), and **expected battle frame** (`target_delay` = F_b) on the expedition, then saves.

In [8]:
x.select_target()

[expedition] 11:17:50  === select_target ===



Select by  [t] top ranking   [s] specific starting time  (blank to cancel):  s


Enter a starting time (year ignored). e.g. '2000-05-30 14:59:59' or '05-30 14:59:59'.


Starting time (blank to cancel):  2000-07-24 14:45:55


  -> best target for 2000-07-24 14:45:55: M=327919 ms, F_b=20045, P~28.7%
[expedition] Saved to data/expeditions/metang.json
[expedition] 11:18:22  target set: boot 2000-07-24T14:45:55, timer M=327919 ms, expected F_b=20045 (P~28.7%). Saved.


{'rank': 1629,
 'initial_time': '2000-07-24T14:45:55',
 'M': 327919,
 'target_delay': 20045,
 'second': 333,
 'p': 0.28714917806215257,
 'sigma': 71.9410734754358,
 'mdmsh': [247, 14]}

## Examine target area

In [9]:
 x.check().chart_check_target_landing()


chart_check_target_landing  (jitter kernel, k=3.5)
boot=2000-07-24T14:45:55  timer M=327919 ms  ->  mean F_b=20045.0 (target_delay=20045)  sigma=71.9
target second=333  mdmsh(m,h)=(247, 14)  window frames [19793, 20297] (505)
  frame      Δ        seed  hit    weight        w%      cumP%
--------------------------------------------------------------
  19793   -252  0xF70E4D51    ✗    0.0022    0.001%     0.000%
  19794   -251  0xF70E4D52    ✓    0.0023    0.001%     0.001%
  19795   -250  0xF70E4D53    ✗    0.0024    0.001%     0.001%
  19796   -249  0xF70E4D54    ✗    0.0025    0.001%     0.001%
  19797   -248  0xF70E4D55    ✗    0.0026    0.001%     0.001%
  19798   -247  0xF70E4D56    ✗    0.0028    0.002%     0.001%
  19799   -246  0xF70E4D57    ✓    0.0029    0.002%     0.003%
  19800   -245  0xF70E4D58    ✗    0.0030    0.002%     0.003%
  19801   -244  0xF70E4D59    ✗    0.0032    0.002%     0.003%
  19802   -243  0xF70E4D5A    ✓    0.0033    0.002%     0.005%
  19803   -242  0

{'p': 0.28714901963226264,
 'n_frames': 505,
 'n_captured': 121,
 'mismatches': None}

# Compass - Identify target

## Calibrate using metronome

In [ ]:
x.metronome_compass()
x.save()

## Finding what seed you hit in safari

`compass_safari()` builds candidates from the calibrated model: for the commanded countdown **M** (set by `select_target`) it sweeps the battle-frame window **F\* ± kσ** across second offsets **δ∈{−1,0,+1}** (off-by-one timer-start timing — each δ uses the *same* frame window). No hand-set delay window.

As you enter observed turns it ranks survivors by **posterior landing probability** (`P(land)`), shows the most-likely seed and which **δ** you hit ("timer on time / +1s late"), and flags when one candidate passes the confidence threshold. Extra commands:

- **`w`** — widen the frame (`k`) and/or second (`±K`) window and re-apply your path so far (also offered automatically on a no-match).
- The set is bounded to the seeds carrying `mass_cap` (default 0.999) of the landing probability; the Jane offload tip triggers on the *prior-weighted* effective count.

Pass `second_offsets=` / `mass_cap=` to override. Afterwards, `x.save_safari_run()` logs the identified seed, observed path, and inferred timer offset to `data/safari_runs.jsonl` (no capture required) for future model retuning.

In [ ]:
x.compass_safari()
x.save()

In [ ]:
# Loop-back: log this run (seed, observed path, inferred timer offset) for model retuning.
# No capture required — records even a fled/ambiguous run.
x.save_safari_run()

# Machete - Finding a path through seed

This is usually triggered during the "Finding what seed you hit in safari" step, but here's some manual activation anyways

## Finding a path for a single seed

In [ ]:
x.machete_one(max_turns=1000)
x.save()